# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bgrivero/flyrank-ml-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


**Lane 1: Ranking Signal Analysis. Task type: scoring (a continuous score per page), used downstream to rank a review queue.**

The lane question is "which safe content and search signals are associated with visibility, clicks, engagement, or movement?" That is not a yes/no decision (classification), nor is it looking for natural groups of similar pages (clustering). A pure ranking needs a metric to sort by, so the actual model output is a **score**; a number, built from content and search signals, that says roughly how strong this page's engagement looks. Then, the ranking signal analysis will be accomplished using this score.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `engagement_rate`** — `engaged_sessions_90d / sessions_90d × 100`, aggregated per page over the trailing 90 days.

This is an **observed measurement**, not a defined rule. GA4 logged an actual session as "engaged" or not (based on GA4's own engagement definition — dwell time, scroll depth, or a conversion event), and the column is a straight ratio of two counted things that already happened. I am not writing an if-statement to decide what counts as good engagement — I am using a number the analytics platform already measured.


In [6]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Brief check for engagement rate formula
check = (df["engaged_sessions_90d"] / df["sessions_90d"] * 100).round(2)
matches = (check.fillna(0) - df["engagement_rate"].fillna(0)).abs().lt(0.01).mean()
print(f"engagement_rate matches engaged_sessions_90d / sessions_90d * 100 for "
      f"{matches:.1%} of rows")

print()
print(df["engagement_rate"].describe())


engagement_rate matches engaged_sessions_90d / sessions_90d * 100 for 100.0% of rows

count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Spearman rank correlation (ρ) between a candidate signal-based score and observed `engagement_rate`**

We look rank correlation because of what the output is *for*: an editor working a review queue cares whether the pages at the **top** of my ranking really are the ones with the weakest engagement relative to their traffic. We are not asking whether we can predict the exact engagement-rate number, e.g. with Pearson's. Spearman ρ score exactly that: does the order I produce match the order reality produced, regardless of scale. A ρ near 0 means my signals tell you nothing about who to review first; a ρ meaningfully above 0 (and holding up on pages the score didn't see) means the ranking is worth something.

In [8]:
from scipy.stats import spearmanr

# Sample baseline score using impressions_90d
baseline_score = df["impressions_90d"]

rho, p_value = spearmanr(baseline_score, df["engagement_rate"])
print(f"Baseline (impressions-only) rank correlation with engagement_rate: "
      f"rho = {rho:.3f} (p = {p_value:.3g})")


Baseline (impressions-only) rank correlation with engagement_rate: rho = 0.487 (p = 0)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page), aggregated over its trailing 90-day window, for one pseudonymized client.** `content_id` is the grain; `client_id` groups pages that belong to the same site (used later for client-holdout splits, never as a feature).

In [9]:
unit_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "word_count", "content_age_days", "days_since_last_update", "freshness_tier",
    "impressions_90d", "clicks_90d", "sessions_90d", "ctr", "avg_position", "position_tier",
    "scroll_rate", "ai_traffic_pct", "engagement_rate",
]

print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"distinct pages (content_id): {df['content_id'].nunique():,}")
print(f"distinct clients (client_id): {df['client_id'].nunique()}")

df[unit_cols].head(8)


shape: 30,000 rows x 44 columns
distinct pages (content_id): 30,000
distinct clients (client_id): 32


,content_id,client_id,content_type,main_intent,word_count,content_age_days,days_since_last_update,freshness_tier,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,position_tier,scroll_rate,ai_traffic_pct,engagement_rate
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,187,20,0-30,3803,29,17,0.76,10.6,striking,4.55,0.0,5.88
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,445,25,0-30,15320,7,9,0.05,20.3,page_3_5,10.00,0.0,0.00
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,141,20,0-30,12581,11,11,0.09,36.5,page_3_5,28.57,0.0,0.00
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,463,22,0-30,11751,58,78,0.49,6.2,page_1,3.45,0.0,1.28
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,263,14,0-30,19140,24,145,0.13,44.0,page_3_5,24.29,0.0,0.00
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3080.0,147,20,0-30,3970,1,5,0.03,8.5,page_1,25.00,0.0,0.00
6,content_9a34b442b552,client_8722616204,keyword article,informational,3059.0,90,20,0-30,20,0,1,0.00,7.0,page_1,0.00,0.0,0.00
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,NaN,445,22,0-30,1724,1,28,0.06,21.2,page_3_5,7.14,0.0,3.57


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



In [11]:
num_cols = ["word_count", "content_age_days", "days_since_last_update",
            "avg_position", "ctr", "scroll_rate", "ai_traffic_pct", "search_volume"]

print("Pairwise correlation with engagement_rate (all individually weak):")
print(df[num_cols + ["engagement_rate"]].corr(numeric_only=True)["engagement_rate"]
      .drop("engagement_rate").sort_values(key=abs, ascending=False).round(3))

print()
print("Mean engagement_rate by position_tier (results betray intuition, top_3 < deep):")
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
print(df.groupby("position_tier")["engagement_rate"].mean().reindex(order).round(2))



Pairwise correlation with engagement_rate (all individually weak):
scroll_rate               0.163
ctr                       0.097
word_count               -0.063
content_age_days          0.040
ai_traffic_pct            0.032
avg_position             -0.018
days_since_last_update   -0.014
search_volume            -0.010
Name: engagement_rate, dtype: float64

Mean engagement_rate by position_tier (results betray intuition, top_3 < deep):
position_tier
top_3       1.75
page_1      2.93
striking    2.48
page_3_5    2.20
deep        2.45
Name: engagement_rate, dtype: float64


A fixed rule needs a signal that moves in one direction, cleanly, on its own. Looking at `engagement_rate` against the signals I actually have, none of them behave that way:

- **No signal is individually strong.** Every pairwise correlation below is weak (|r| well under 0.2). As such, there is no single column an if-statement could threshold on and get a useful split.
- **The obvious rule is backwards.** A reasonable first guess is "better search position → higher engagement." The grouped means below show `top_3` pages have the *lowest* average engagement rate of any position tier, and `page_1` (not `top_3`) has the highest. A hand-written rule built on that intuition would misfire on exactly the pages it's meant to catch.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.